# 联动标识检查

本 notebook 用于检查《备件_不联动清单_整改及跟踪项.xlsx》与《eps3_spare.xlsx》中联动标识不一致的情况。

In [ ]:
import pandas as pd

# 文件路径
file1 = '备件_不联动清单_整改及跟踪项.xlsx'
file2 = 'eps3_spare.xlsx' 

## 步骤1：读取《备件_不联动清单_整改及跟踪项.xlsx》，筛选对策为"不联动→联动（已确定）"的记录，提取 key1

In [ ]:
# 读取第一个文件
df1 = pd.read_excel(file1)

# 查看前几行（可选）
# df1.head()

# 筛选"对策"为"不联动→联动（已确定）"的记录
df1_filtered = df1[df1['对策'] == '不联动→联动（已确定）'].copy()

# 构造 key1：备件号 + 供应商编码
df1_filtered['key1'] = df1_filtered['备件号'].astype(str) + '_' + df1_filtered['供应商编码'].astype(str)

# 去重后得到 key1 集合
key1_set = set(df1_filtered['key1'].unique())

print(f'步骤1：key1 数量 = {len(key1_set)}')
df1_filtered.head()

## 步骤2：读取《eps3_spare.xlsx》，筛选通知书状态为"已审批"的记录，提取 key2

In [ ]:
# 读取第二个文件
df2 = pd.read_excel(file2)

# 查看前几行（可选）
# df2.head()

# 筛选"通知书状态"为"已审批"的记录
df2_filtered = df2[df2['通知书状态'] == '已审批'].copy()

# 构造 key2：备件号 + 供应商编码
df2_filtered['key2'] = df2_filtered['备件号'].astype(str) + '_' + df2_filtered['供应商编码'].astype(str)

print(f'步骤2：已审批记录数量 = {len(df2_filtered)}')
df2_filtered.head()

## 步骤3：对 eps3_spare 已审批记录按同一备件号、同一供应商编码分类
如果"生效日期"与"失效日期"不在同一年，先将"失效日期"调整为"生效日期"当年的最后一天，再按调整后的 失效日期 降序、再按 创建时间 降序，取每组第一条（即失效日期最晚，相同时取创建时间最晚）

In [ ]:
# 确保日期列为 datetime 类型
df2_filtered['生效日期'] = pd.to_datetime(df2_filtered['生效日期'], errors='coerce')
df2_filtered['失效日期'] = pd.to_datetime(df2_filtered['失效日期'], errors='coerce')
df2_filtered['创建时间'] = pd.to_datetime(df2_filtered['创建时间'], errors='coerce')

# 如果"生效日期"与"失效日期"不在同一年，将"失效日期"调整为"生效日期"当年的最后一天
def adjust_expiry(row):
    eff = row['生效日期']
    exp = row['失效日期']
    if pd.notna(eff) and pd.notna(exp) and eff.year != exp.year:
        return pd.Timestamp(year=eff.year, month=12, day=31)
    return exp

df2_filtered['失效日期_调整后'] = df2_filtered.apply(adjust_expiry, axis=1)

# 按 key2 分组，先按 失效日期_调整后 降序，再按 创建时间 降序，取第一条
df2_grouped = (
    df2_filtered
    .sort_values(['key2', '失效日期_调整后', '创建时间'], ascending=[True, False, False])
    .groupby('key2', as_index=False)
    .first()
)

print(f'步骤3：分组后记录数量 = {len(df2_grouped)}')
df2_grouped.head()

## 步骤4：匹配 key1 与 key2 相同的记录，将 key2 的"价格通知书号"、"备件裸价"、"备件物流费"、"备件包装费"、"是否联动(0是，1否)"添加到 key1 对应的记录中

In [ ]:
# 从 key2 分组结果中选取需要合并的列
df2_merge = df2_grouped[['key2', '价格通知书号', '备件裸价', '备件物流费', '备件包装费', '是否联动(0是，1否)']].copy()

# 以 key1（df1_filtered）为基准，左连接 key2 的对应字段
df_result = df1_filtered.merge(
    df2_merge,
    how='left',
    left_on='key1',
    right_on='key2'
)

print(f'步骤4：key1 总记录数 = {len(df_result)}')
print(f'      成功匹配 key2 的记录数 = {df_result["key2"].notna().sum()}')

# 如需仅保留匹配成功的记录，可取消下一行注释
# df_result = df_result[df_result['key2'].notna()].copy()

df_result.head()

## 结果导出（可选）

In [ ]:
from datetime import datetime
today = datetime.now().strftime("%Y%m%d")

# 导出结果到 Excel（如需）
output_file = f'./联动标识检查结果/94项需改为联动标识检查结果_{today}.xlsx'
df_result.to_excel(output_file, index=False)
print(f'结果已导出到 {output_file}')